In [46]:
import pandas as pd
import glob
import os
from collections import defaultdict

RAW_DIR = "../data/raw"
SUMMARY_DIR = "../data/timeseries_summary"
PROCESSED_DIR = "../data/processed"

os.makedirs(SUMMARY_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

csv_files = sorted(glob.glob(os.path.join(RAW_DIR, "*.csv")))

print("Number of files:", len(csv_files))
csv_files[:3]

Number of files: 20


['../data/raw\\clean_20181024_dX_0830_0900.csv',
 '../data/raw\\clean_20181024_dX_0900_0930.csv',
 '../data/raw\\clean_20181024_dX_0930_1000.csv']

In [47]:
sample = pd.read_csv(csv_files[0], nrows=5)

sample.head()
sample.columns.tolist()

['track_id',
 'type',
 'traveled_d',
 'avg_speed',
 'lat',
 'lon',
 'speed',
 'lon_acc',
 'lat_acc',
 'time',
 'timestamp_real',
 'time_str',
 'time_bin_5m',
 'unique_track_id',
 'grid_id',
 'is_crawling',
 'is_hard_braking',
 'pce_factor']

In [48]:
PCE_DICT = {
    "Motorcycle": 0.5,
    "Bicycle": 0.5,
    "Car": 1.0,
    "Taxi": 1.0,
    "Medium Vehicle": 1.5,
    "Heavy Vehicle": 2.5,
    "Bus": 2.5,
}

use_cols = [
    "time_bin_5m",
    "unique_track_id",
    "speed",
    "type",
    "is_crawling",
    "is_hard_braking"
]

In [49]:
all_summaries = []

for file in csv_files:
    print("Processing:", os.path.basename(file))

    vehicle_chunks = []

    for chunk in pd.read_csv(file, usecols=use_cols, chunksize=1_000_000):
        chunk["time_bin_5m"] = pd.to_datetime(chunk["time_bin_5m"])

        chunk["type"] = chunk["type"].astype(str).str.strip()

        # Bỏ pedestrian vì không phải phương tiện cơ giới
        chunk = chunk[chunk["type"] != "Pedestrian"].copy()

        if chunk.empty:
            continue

        # Tạo pce_factor từ type
        chunk["pce_factor"] = chunk["type"].map(PCE_DICT)

        if chunk["pce_factor"].isna().any():
            print("Unknown vehicle types in:", os.path.basename(file))
            print(chunk.loc[chunk["pce_factor"].isna(), "type"].value_counts())

        chunk["pce_factor"] = chunk["pce_factor"].fillna(1.0)

        vehicle_level = chunk.groupby(
            ["time_bin_5m", "unique_track_id"]
        ).agg(
            avg_speed_vehicle=("speed", "mean"),
            pce_factor=("pce_factor", "first"),
            is_crawling=("is_crawling", "max"),
            is_hard_braking=("is_hard_braking", "max")
        ).reset_index()

        vehicle_chunks.append(vehicle_level)

    if len(vehicle_chunks) == 0:
        print("No valid vehicle data in:", os.path.basename(file))
        continue

    vehicle_data = pd.concat(vehicle_chunks, ignore_index=True)

    vehicle_data = vehicle_data.groupby(
        ["time_bin_5m", "unique_track_id"]
    ).agg(
        avg_speed_vehicle=("avg_speed_vehicle", "mean"),
        pce_factor=("pce_factor", "first"),
        is_crawling=("is_crawling", "max"),
        is_hard_braking=("is_hard_braking", "max")
    ).reset_index()

    file_summary = vehicle_data.groupby("time_bin_5m").agg(
        vehicle_count=("unique_track_id", "nunique"),
        avg_speed=("avg_speed_vehicle", "mean"),
        pce_volume=("pce_factor", "sum"),
        crawling_count=("is_crawling", "sum"),
        hard_braking_count=("is_hard_braking", "sum")
    ).reset_index()

    file_summary = file_summary.sort_values("time_bin_5m")

    out_name = "timeseries_" + os.path.basename(file)
    out_path = os.path.join(SUMMARY_DIR, out_name)

    file_summary.to_csv(out_path, index=False)

    all_summaries.append(file_summary)

print("Done")

Processing: clean_20181024_dX_0830_0900.csv
Processing: clean_20181024_dX_0900_0930.csv
Processing: clean_20181024_dX_0930_1000.csv
Processing: clean_20181024_dX_1000_1030.csv
Processing: clean_20181024_dX_1030_1100.csv
Processing: clean_20181029_dX_0800_0830.csv
Processing: clean_20181029_dX_0830_0900.csv
Processing: clean_20181029_dX_0900_0930.csv
Processing: clean_20181029_dX_0930_1000.csv
Processing: clean_20181029_dX_1000_1030.csv
Processing: clean_20181030_dX_0800_0830.csv
Processing: clean_20181030_dX_0830_0900.csv
Processing: clean_20181030_dX_0900_0930.csv
Processing: clean_20181030_dX_0930_1000.csv
Processing: clean_20181030_dX_1000_1030.csv
Processing: clean_20181101_dX_0800_0830.csv
Processing: clean_20181101_dX_0830_0900.csv
Processing: clean_20181101_dX_0900_0930.csv
Processing: clean_20181101_dX_0930_1000.csv
Processing: clean_20181101_dX_1000_1030.csv
Done


In [51]:
traffic_ts = pd.concat(all_summaries, ignore_index=True)

traffic_ts = traffic_ts.groupby("time_bin_5m").agg(
    vehicle_count=("vehicle_count", "sum"),
    avg_speed=("avg_speed", "mean"),
    pce_volume=("pce_volume", "sum"),
    crawling_count=("crawling_count", "sum"),
    hard_braking_count=("hard_braking_count", "sum")
).reset_index()

traffic_ts = traffic_ts.sort_values("time_bin_5m")

out_file = os.path.join(PROCESSED_DIR, "traffic_density_timeseries.csv")

traffic_ts.to_csv(out_file, index=False)

print("Saved to:", out_file)
traffic_ts.head()

Saved to: ../data/processed\traffic_density_timeseries.csv


,time_bin_5m,vehicle_count,avg_speed,pce_volume,crawling_count,hard_braking_count
0,2018-10-24 08:30:00,4724,17.826713,4383.0,3427,1419
1,2018-10-24 08:35:00,5106,17.355170,4733.0,3816,1654
2,2018-10-24 08:40:00,3905,16.790361,3633.5,2907,1174
3,2018-10-24 09:00:00,4620,14.779216,4308.0,3809,1441
4,2018-10-24 09:05:00,4983,14.968209,4627.0,4016,1578
